# SAM 2 — sinh mask 1 lần cho TẤT CẢ scene, dùng chung cho mọi model

Notebook chạy **Stage 1 của Split&Splat** (https://github.com/LTTM/Split_and_Splat, file `sam2/auto_seg.py`) trên **mọi scene** trong `/content/data`:
SAM 2.1 (bản `large`) tạo mask tự động với **4 cấu hình** (`uber_huge` → `very_coarse` → `coarse` → `fine`) cho **từng ảnh**, rồi gộp và lọc.
Mỗi scene ra **một thư mục riêng** để SuGaR / GOF / GSurf / Split&Splat... đọc chung, không phải chạy SAM lại.

```
Google Drive (gdown) → giải nén → tự tìm các scene → [SAM 2: 4 cấu hình → gộp → lọc] → mask 2D / frame → zip từng scene → tải về máy
                                                                ▲ notebook này        (Stage 2: + depth + pose → ID nhất quán: làm sau)
```

**Cách dùng:** sửa ô *"1. Tham số đầu vào"* rồi `Runtime ▸ Run all`. Cần runtime **GPU — khuyên dùng L4** (có bfloat16 như môi trường gốc).
Thư viện cài bằng `uv`, ghim đúng phiên bản của tác giả (mục 2).

### Thư mục kết quả (`OUTPUT_ROOT/<scene>/`)

| Đường dẫn | Nội dung |
|---|---|
| `03_final/masks/<frame>/<k>.png` | **Mask cuối cùng**, nhị phân 0/255, mỗi mask 1 file. Cùng định dạng thư mục `output/<scene>_autoseg_mask/<frame>/<id>.png` mà code gốc của tác giả đọc ở Stage 2 |
| `03_final/id_map/<frame>.png` | 1 ảnh uint16 / frame: 0 = nền, `k+1` = mask `k`. Chỗ mask chồng nhau thì mask **nhỏ hơn đè lên** |
| `03_final/npz/<frame>.npz` | `masks` (N,H,W) bool + `area_px` + `bbox_xyxy`, nạp 1 lệnh bằng numpy |
| `03_final/meta/<frame>.json` | Thông tin từng frame. Ghi **sau cùng** nên dùng làm dấu "frame đã xong" để chạy tiếp khi Colab đứt |
| `04_overlay/<frame>.png` | Ảnh gốc + mask tô màu + số ID, để nhìn bằng mắt |
| `01_per_config/<frame>/<cfg>/<k>.png` | *(tuỳ chọn)* mask thô của từng cấu hình SAM trước khi gộp |
| `02_merged/<frame>/<k>.png` | *(tuỳ chọn)* mask sau khi gộp, trước bước lọc cuối |
| `manifest.json` | Danh sách toàn bộ frame + đường dẫn file — **file các model khác nên đọc đầu tiên** |
| `instances.csv` / `stats.csv` | Mỗi mask 1 dòng / mỗi frame 1 dòng |
| `run_info.json` | Checkpoint, 4 cấu hình SAM, ngưỡng, phiên bản thư viện, thời gian chạy |
| `palette.json` | Bảng màu cố định theo ID |

Ngoài ra `OUTPUT_ROOT/summary.csv` tổng hợp mọi scene (số frame, số mask, thời gian, VRAM).

> ⚠️ **ID chỉ có nghĩa trong 1 frame.** Mask số 3 ở frame A và mask số 3 ở frame B không phải cùng một vật.
> Muốn ID nhất quán giữa các view (để dựng 3DGS riêng từng object) cần Stage 2 (lan truyền mask bằng depth + pose + point cloud) — xem mục cuối.

## 0. Kiểm tra GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Chưa có GPU: Runtime ▸ Change runtime type ▸ chọn GPU (khuyên dùng L4) rồi chạy lại."
print("GPU:", gpu)   # L4 = compute capability 8.9 → có bfloat16 phần cứng như môi trường gốc

## 1. Tham số đầu vào *(sửa ở đây)*

In [ ]:
#@title 1. Tham số đầu vào { display-mode: "form" }

#@markdown ### Dữ liệu (tải từ Google Drive bằng gdown)
DOWNLOAD_DATA = True  #@param {type:"boolean"}
GDRIVE_IDS = "1t08_1xV1h5pCdU6-oKRP49veUjxUSM_8, 1LqmUsixkDkw7GIWQX0ZmJxXjFdMMlPjc"  #@param {type:"string"}
DATA_ROOT  = "/content/data"  #@param {type:"string"}
#@markdown `GDRIVE_IDS`: ID các file zip trên Drive, cách nhau bằng dấu phẩy. Zip nào đã có sẵn trong `DATA_ROOT` thì dùng luôn (bạn cũng có thể tự upload zip vào đó và đặt `DOWNLOAD_DATA = False`).

#@markdown ### Chọn scene
SCENES       = "all"     #@param {type:"string"}
IMAGE_SUBDIR = "images"  #@param ["images", "images_2", "images_4", "images_8"] {allow-input: true}
IMAGE_EXTS   = ".jpg,.jpeg,.png"  #@param {type:"string"}
#@markdown `SCENES`: `all` = mọi scene tìm thấy trong `DATA_ROOT`, hoặc liệt kê tên cách nhau bằng dấu phẩy (vd `teatime, figurines`).
#@markdown Notebook coi **mỗi thư mục tên `IMAGE_SUBDIR` là ảnh của 1 scene**, tên scene = tên thư mục cha. Nên chạy trên `images` (độ phân giải cao nhất).
IMAGE_DIR  = ""  #@param {type:"string"}
SCENE_NAME = ""  #@param {type:"string"}
#@markdown *(Tuỳ chọn)* điền `IMAGE_DIR` (+ `SCENE_NAME`) để chạy riêng đúng 1 thư mục ảnh, bỏ qua bước tự tìm scene.

#@markdown ### Kết quả
OUTPUT_ROOT = "/content/outputs/masks"  #@param {type:"string"}
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown Mặc định ghi ra `/content` rồi tải về máy (**hết phiên Colab là mất**). Bật `USE_GOOGLE_DRIVE` nếu muốn mount Drive (đặt `OUTPUT_ROOT` trong `/content/drive/MyDrive/...`).

#@markdown ### Chọn frame (áp dụng cho từng scene)
FRAME_STRIDE = 1  #@param {type:"integer"}
MAX_FRAMES   = 0  #@param {type:"integer"}
#@markdown `FRAME_STRIDE`: lấy 1 frame mỗi N frame. `MAX_FRAMES`: 0 = tất cả, >0 để chạy thử (vd 5).

#@markdown ### SAM 2
SAM2_MODEL = "large"  #@param ["tiny", "small", "base_plus", "large"]
AUTOCAST   = "bfloat16"  #@param ["bfloat16", "auto", "float16"]
#@markdown `bfloat16` = đúng code gốc (L4/A100 chạy được). `auto`: bfloat16 nếu GPU ≥ Ampere, còn lại float16 (T4 không có bfloat16 phần cứng nên kết quả sẽ lệch nhẹ).

#@markdown ### Môi trường (cài bằng uv)
PIN_AUTHOR_ENV = True  #@param {type:"boolean"}
SAM2_GIT_REF   = "main"  #@param {type:"string"}
#@markdown `PIN_AUTHOR_ENV`: ghim đúng phiên bản trong `environment.yml` của tác giả (torch 2.5.1+cu124, torchvision 0.20.1, numpy 2.1.3, pillow 11.1.0, hydra-core 1.3.2, iopath 0.1.10). Tác giả không ghim commit của SAM 2 nên `SAM2_GIT_REF` mặc định là `main`; commit thực tế được ghi vào `run_info.json`.

#@markdown ### Gộp & lọc mask (mặc định = code gốc của tác giả)
MERGE_THRESH   = 0.3     #@param {type:"number"}
DUP_THRESH     = 0.95    #@param {type:"number"}
MIN_MASK_RATIO = 0.0025  #@param {type:"number"}
#@markdown Mask nhỏ hơn `MIN_MASK_RATIO` diện tích ảnh bị bỏ; mask chồng nhau > `MERGE_THRESH` (tính theo mask nhỏ hơn) được gộp; chồng > `DUP_THRESH` bị coi là trùng.

#@markdown ### Lưu thêm file trung gian
SAVE_PER_CONFIG = False  #@param {type:"boolean"}
SAVE_MERGED     = False  #@param {type:"boolean"}
RESUME          = True   #@param {type:"boolean"}
#@markdown `RESUME`: bỏ qua frame đã có `meta/<frame>.json` (chạy tiếp sau khi chạy lại ô, nếu runtime còn sống).

#@markdown ### Tải kết quả về máy
DOWNLOAD_WHEN = "after_each_scene"  #@param ["after_each_scene", "at_end", "never"]
ZIP_PARTS     = 1  #@param {type:"integer"}
#@markdown `after_each_scene`: xong scene nào nén và tải scene đó ngay (**an toàn nhất** nếu Colab bị ngắt giữa chừng). `ZIP_PARTS`: chia mỗi scene thành N file zip nếu quá lớn (trình duyệt hay lỗi khi tải file > ~1–2 GB).

## 2. Cài thư viện bằng `uv` (khớp môi trường tác giả)

Môi trường gốc: Python 3.12, **torch 2.5.1 + CUDA 12.4**, torchvision 0.20.1, numpy 2.1.3, pillow 11.1.0, hydra-core 1.3.2, iopath 0.1.10.
torch 2.5.1 chỉ có wheel cho Python ≤ 3.12; nếu Colab đang chạy Python 3.13 thì notebook báo và dùng torch có sẵn của Colab (kết quả có thể lệch nhẹ).
Nếu kernel đang giữ numpy bản khác, ô kiểm tra bên dưới sẽ **tự khởi động lại runtime** — chỉ cần bấm `Run all` thêm một lần (lần 2 uv thấy đã đủ nên rất nhanh).

In [ ]:
import sys, subprocess
PY = sys.version_info[:2]
CAN_PIN = bool(PIN_AUTHOR_ENV) and PY <= (3, 12)
print("Python", sys.version.split()[0], "| ghim môi trường tác giả:", CAN_PIN)
if PIN_AUTHOR_ENV and not CAN_PIN:
    print("⚠️ Python > 3.12 không có wheel torch 2.5.1 → bỏ qua bước ghim, dùng torch có sẵn của Colab.")

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-1500:])
    assert r.returncode == 0, f"Lệnh lỗi (mã {r.returncode})"

sh("pip -q install uv")
if CAN_PIN:
    sh("uv pip install --system -q torch==2.5.1 torchvision==0.20.1")
    sh("uv pip install --system -q numpy==2.1.3 pillow==11.1.0 hydra-core==1.3.2 iopath==0.1.10")
# SAM 2 (bản chính thức của Meta). Bỏ build CUDA ext _C: chỉ dùng cho hậu xử lý lấp lỗ, mà code gốc dùng apply_postprocessing=False.
sh("uv pip install --system -q setuptools wheel")
sh(f'SAM2_BUILD_CUDA=0 uv pip install --system -q --no-build-isolation "git+https://github.com/facebookresearch/sam2.git@{SAM2_GIT_REF}"')

In [ ]:
# Kernel có thể đã nạp numpy/torch bản cũ trước khi uv đổi phiên bản → khởi động lại 1 lần cho chắc.
import os, numpy
need_restart = CAN_PIN and numpy.__version__ != "2.1.3"
if not need_restart and CAN_PIN:
    import torch
    need_restart = not torch.__version__.startswith("2.5.1")
if need_restart:
    print("Đã cài đúng phiên bản nhưng kernel đang giữ bản cũ → tự khởi động lại runtime.\n"
          "Khi runtime báo đã ngắt, bấm Runtime ▸ Run all lại (lần này uv bỏ qua phần đã cài).")
    os.kill(os.getpid(), 9)
import torch, torchvision
print("OK | torch", torch.__version__, "| torchvision", torchvision.__version__, "| numpy", numpy.__version__,
      "| CUDA", torch.version.cuda, "| GPU sẵn sàng:", torch.cuda.is_available())
assert torch.cuda.is_available(), "torch không thấy GPU."

## 3. Tải dữ liệu từ Drive, giải nén, tìm scene

- `gdown` bản mới **đã bỏ cờ `--id`**, nên ID được truyền trực tiếp (`gdown <ID>`).
- Zip nào giải nén vào `DATA_ROOT/<tên zip>/`. Rác macOS (`__MACOSX`, `.DS_Store`) được bỏ qua.
- Drive đôi khi báo "quota exceeded / too many users" với file công khai: khi đó tải lại sau ít phút, hoặc chép file vào Drive của bạn rồi upload zip trực tiếp vào `DATA_ROOT`.

In [ ]:
import re, json, csv, time, shutil, os, sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

DATA_ROOT = Path(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# --- tải ---
ids = [x.strip() for x in GDRIVE_IDS.replace("\n", ",").split(",") if x.strip()]
have_zips = list(DATA_ROOT.glob("*.zip"))
if DOWNLOAD_DATA and not IMAGE_DIR:
    if len(have_zips) >= len(ids) > 0:
        print(f"Đã có {len(have_zips)} file zip trong {DATA_ROOT} → bỏ qua bước tải.")
    else:
        sh("pip -q install gdown")
        for fid in ids:
            sh(f"gdown {fid} -O {DATA_ROOT}/")
    have_zips = list(DATA_ROOT.glob("*.zip"))

# --- giải nén ---
def unzip_clean(zp, dest):
    with zipfile.ZipFile(zp) as zf:
        members = [m for m in zf.namelist() if "__MACOSX" not in m and not m.endswith(".DS_Store")]
        zf.extractall(dest, members)
    return len(members)

for zp in sorted(have_zips):
    dest = DATA_ROOT / zp.stem
    if dest.is_dir() and any(dest.iterdir()):
        print(f"{zp.name}: đã giải nén ({dest}) → bỏ qua")
        continue
    n = unzip_clean(zp, dest)
    print(f"{zp.name}: giải nén {n} mục → {dest}")

In [ ]:
exts = tuple(e.strip().lower() for e in IMAGE_EXTS.split(",") if e.strip())
want = None if SCENES.strip().lower() == "all" else {s.strip() for s in SCENES.split(",") if s.strip()}

def list_images(d):
    return sorted((p for p in Path(d).iterdir() if p.suffix.lower() in exts), key=natural_key)

# 1) tìm (tên scene → thư mục ảnh)
if IMAGE_DIR:
    d = Path(IMAGE_DIR)
    assert d.is_dir(), f"Không thấy thư mục ảnh: {d}"
    candidates = {(SCENE_NAME or d.parent.name): d}
else:
    candidates = {}
    for d in sorted(DATA_ROOT.rglob(IMAGE_SUBDIR)):
        if not d.is_dir() or "__MACOSX" in d.parts or not list_images(d):
            continue
        name = d.parent.name
        if name in candidates:
            print(f"⚠️ Trùng tên scene '{name}': bỏ {d} (giữ {candidates[name]})")
            continue
        candidates[name] = d
assert candidates, f"Không tìm thấy thư mục '{IMAGE_SUBDIR}' có ảnh nào trong {DATA_ROOT}"
if want is not None:
    missing = want - set(candidates)
    assert not missing, f"Không có scene {sorted(missing)}; tìm thấy: {sorted(candidates)}"
    candidates = {k: v for k, v in candidates.items() if k in want}

# 2) dựng danh sách scene
SCENES_LIST = []
for name in sorted(candidates):
    image_dir = candidates[name]
    all_images = list_images(image_dir)
    images = all_images[::max(1, FRAME_STRIDE)]
    if MAX_FRAMES > 0:
        images = images[:MAX_FRAMES]
    stems = [p.stem for p in images]
    dup = {s for s in stems if stems.count(s) > 1}
    assert not dup, f"[{name}] trùng tên (khác đuôi) sẽ ghi đè nhau: {sorted(dup)[:5]}"
    out = Path(OUTPUT_ROOT) / name
    for sub in ["03_final/masks", "03_final/id_map", "03_final/npz", "03_final/meta", "04_overlay"] \
               + (["01_per_config"] if SAVE_PER_CONFIG else []) + (["02_merged"] if SAVE_MERGED else []):
        (out / sub).mkdir(parents=True, exist_ok=True)
    SCENES_LIST.append({"name": name, "image_dir": image_dir, "images": images, "stems": stems, "out": out,
                        "n_all": len(all_images)})

rows = []
for sc in SCENES_LIST:
    w, h = Image.open(sc["images"][0]).size
    siblings = sorted(p.name for p in sc["image_dir"].parent.iterdir() if p.is_dir() and p.name.startswith("images"))
    rows.append({"scene": sc["name"], "ảnh_tổng": sc["n_all"], "sẽ_xử_lý": len(sc["images"]), "kích_thước": f"{w}x{h}",
                 "thư_mục_ảnh": str(sc["image_dir"]), "thư_mục_ảnh_cùng_cấp": ", ".join(siblings)})
display(pd.DataFrame(rows).set_index("scene"))
print(f"\nTổng: {len(SCENES_LIST)} scene, {sum(len(s['images']) for s in SCENES_LIST)} ảnh × 4 cấu hình SAM")
print("Thư mục ra:", OUTPUT_ROOT)

## 4. Tải checkpoint SAM 2

In [ ]:
import urllib.request

CKPT_BASE = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
MODELS = {   # tên → (file checkpoint, file config hydra trong gói sam2)
    "tiny":      ("sam2.1_hiera_tiny.pt",      "configs/sam2.1/sam2.1_hiera_t.yaml"),
    "small":     ("sam2.1_hiera_small.pt",     "configs/sam2.1/sam2.1_hiera_s.yaml"),
    "base_plus": ("sam2.1_hiera_base_plus.pt", "configs/sam2.1/sam2.1_hiera_b+.yaml"),
    "large":     ("sam2.1_hiera_large.pt",     "configs/sam2.1/sam2.1_hiera_l.yaml"),
}
ckpt_name, MODEL_CFG = MODELS[SAM2_MODEL]
CKPT_PATH = Path("/content/checkpoints") / ckpt_name
CKPT_PATH.parent.mkdir(exist_ok=True)
if not CKPT_PATH.exists():
    print("Đang tải", ckpt_name, "…")
    urllib.request.urlretrieve(CKPT_BASE + ckpt_name, CKPT_PATH)
print(CKPT_PATH, f"{CKPT_PATH.stat().st_size / 1e6:.0f} MB")

## 5. Cấu hình SAM 2 *(4 cấu hình của tác giả — sửa được)*

Chép nguyên từ khối `__main__` của `sam2/auto_seg.py`. Từ thô đến mịn: `uber_huge` (1 điểm/ảnh, mask rất lớn) → `very_coarse` (4×4 điểm) → `coarse` (8×8) → `fine` (16×16).
Các cấu hình được **chạy theo thứ tự này** và gộp dần; đổi thứ tự hoặc xoá bớt sẽ đổi kết quả. Cả 4 cấu hình đều chạy cho **mọi ảnh của mọi scene**.

In [ ]:
SAM_CONFIGS = [
    {"name": "uber_huge",   "points_per_side": 1,  "points_per_batch": 1,  "pred_iou_thresh": 0.8,
     "stability_score_thresh": 0.85, "stability_score_offset": 0.85, "mask_threshold": 0.5,
     "crop_n_layers": 1, "box_nms_thresh": 0.3, "use_m2m": False},
    {"name": "very_coarse", "points_per_side": 4,  "points_per_batch": 4,  "pred_iou_thresh": 0.8,
     "stability_score_thresh": 0.9,  "stability_score_offset": 0.85, "mask_threshold": 0.5,
     "crop_n_layers": 1, "box_nms_thresh": 0.3, "use_m2m": False},
    {"name": "coarse",      "points_per_side": 8,  "points_per_batch": 8,  "pred_iou_thresh": 0.8,
     "stability_score_thresh": 0.85, "stability_score_offset": 0.85, "mask_threshold": 0.3,
     "crop_n_layers": 1, "box_nms_thresh": 0.5, "use_m2m": False},
    {"name": "fine",        "points_per_side": 16, "points_per_batch": 64, "pred_iou_thresh": 0.8,
     "stability_score_thresh": 0.9,  "stability_score_offset": 1,    "mask_threshold": 0.3,
     "crop_n_layers": 1, "box_nms_thresh": 0.3, "use_m2m": False},
]
CFG_NAMES = [c["name"] for c in SAM_CONFIGS]
display(pd.DataFrame(SAM_CONFIGS).set_index("name"))

## 6. Hàm gộp / lọc mask (từ `auto_seg.py` của tác giả)

In [ ]:
def MaskMerging(merged, mask, merge_thresh=0.3):
    """Nguyên văn auto_seg.py::MaskMerging (chỉ đưa ngưỡng 0.3 thành tham số).
    Mask mới chồng lên mask cũ > merge_thresh (tính theo mask nhỏ hơn) thì gộp (OR), ngược lại thêm mới."""
    mask = sorted(mask, key=lambda x: np.sum(x), reverse=True)
    for m1 in mask:
        merged_flag = False
        for i, m2 in enumerate(merged):
            intersection = np.logical_and(m1, m2).sum()
            if intersection == 0:
                continue
            smaller_area = min(m1.sum(), m2.sum())
            if intersection / smaller_area > merge_thresh:
                merged[i] = np.logical_or(m1, m2)
                merged_flag = True
        if not merged_flag:
            merged.append(m1)
    return merged


def useless_mask(masks, min_ratio=0.0025, iou_threshold=0.95):
    """Nguyên văn auto_seg.py::useless_mask: bỏ mask quá nhỏ và mask trùng (chồng > iou_threshold)."""
    filtered_masks = []
    for m in masks:
        mask_pixels = np.count_nonzero(m)
        if mask_pixels == 0 or mask_pixels / m.size < min_ratio:
            continue
        keep = True
        for fm in filtered_masks:
            intersection = np.logical_and(m, fm).sum()
            smaller_area = min(m.sum(), fm.sum())
            if intersection / smaller_area > iou_threshold:
                keep = False
                break
        if keep:
            filtered_masks.append(m)
    return filtered_masks


def combine_configs(generated):
    """generated: list (mỗi cấu hình thành công 1 phần tử) các list mask. Trả về (merged, final) như getMask() gốc."""
    if len(generated) == 0:
        return [], []
    final = []
    for i, ms in enumerate(generated):
        if i == 0:
            final = useless_mask(sorted(list(ms), key=lambda x: np.sum(x), reverse=True),
                                 MIN_MASK_RATIO, DUP_THRESH)
        else:
            final = MaskMerging(final, ms, MERGE_THRESH)
    merged = list(final)
    return merged, useless_mask(final, MIN_MASK_RATIO, DUP_THRESH)

## 7. Nạp SAM 2

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

major = torch.cuda.get_device_properties(0).major
amp_dtype = {"bfloat16": torch.bfloat16, "float16": torch.float16}.get(
    AUTOCAST, torch.bfloat16 if major >= 8 else torch.float16)
torch.autocast("cuda", dtype=amp_dtype).__enter__()          # giống auto_seg.py (nhưng chọn dtype theo GPU)
if major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
np.random.seed(3)

sam2 = build_sam2(MODEL_CFG, str(CKPT_PATH), device="cuda", apply_postprocessing=False)
print(f"SAM2 {SAM2_MODEL} đã nạp | autocast {amp_dtype} | GPU compute capability {major}.x")

## 8. Hàm xử lý 1 frame / 1 scene

In [ ]:
def id_color(i):
    """Màu cố định cho 1 ID (không đụng RNG toàn cục)."""
    rng = np.random.default_rng(int(i) * 7919 + 12345)
    return tuple(int(x) for x in rng.integers(40, 256, size=3))

def save_bin(mask, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(path)

def bbox_xyxy(m):
    ys, xs = np.nonzero(m)
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

def make_id_map(masks, H, W):
    idmap = np.zeros((H, W), np.uint16)
    for k in sorted(range(len(masks)), key=lambda k: -int(np.count_nonzero(masks[k]))):  # to trước, nhỏ đè lên
        idmap[np.asarray(masks[k]) > 0] = k + 1
    return idmap

def make_overlay(img, masks):
    out = img.astype(np.float32) * 0.45
    layer = np.zeros_like(out)
    covered = np.zeros(img.shape[:2], bool)
    for k in sorted(range(len(masks)), key=lambda k: -int(np.count_nonzero(masks[k]))):
        m = np.asarray(masks[k]) > 0
        layer[m] = id_color(k + 1)
        covered |= m
    out[covered] += layer[covered] * 0.55
    pil = Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))
    draw = ImageDraw.Draw(pil)
    for k, m in enumerate(masks):
        ys, xs = np.nonzero(np.asarray(m) > 0)
        draw.text((int(xs.mean()), int(ys.mean())), str(k + 1), fill=(255, 255, 255), stroke_width=2, stroke_fill=(0, 0, 0))
    return pil

import traceback
_TB_SHOWN = set()

def run_sam_configs(image):
    """Chạy lần lượt các cấu hình SAM. Cấu hình nào ném lỗi thì bị bỏ qua, đúng như getMask() gốc
    (`except: print("NO valid mask!"); continue`). Trả về (generated, per_cfg, failed)."""
    generated, per_cfg, failed = [], {}, []
    for cfg in SAM_CONFIGS:
        gen = SAM2AutomaticMaskGenerator(model=sam2, **{k: v for k, v in cfg.items() if k != "name"})
        try:
            ms = [m["segmentation"].astype(np.uint8) for m in gen.generate(image)]
            generated.append(ms)
            per_cfg[cfg["name"]] = ms
        except Exception as e:
            failed.append(cfg["name"])
            per_cfg[cfg["name"]] = []
            key = (cfg["name"], type(e).__name__, str(e)[:60])
            if key not in _TB_SHOWN:     # in traceback đầy đủ 1 lần cho mỗi loại lỗi để biết lỗi nằm ở dòng nào
                _TB_SHOWN.add(key)
                print(f"  [!] cấu hình {cfg['name']} lỗi → bỏ qua cấu hình này (như nhánh 'NO valid mask!' của tác giả): "
                      f"{type(e).__name__}: {e}")
                print(traceback.format_exc()[-1500:])
            else:
                print(f"  [!] cấu hình {cfg['name']} lỗi (cùng loại lỗi đã in ở trên)")
    return generated, per_cfg, failed

def write_frame(out, stem, img_path, image, per_cfg, merged, final, seconds, failed=()):
    H, W = image.shape[:2]
    fin = out / "03_final"
    mdir = fin / "masks" / stem
    if mdir.exists():
        shutil.rmtree(mdir)
    mdir.mkdir(parents=True, exist_ok=True)       # luôn tạo thư mục, kể cả 0 mask (code gốc listdir thư mục này)
    for k, m in enumerate(final):
        save_bin(m, mdir / f"{k}.png")
    Image.fromarray(make_id_map(final, H, W)).save(fin / "id_map" / f"{stem}.png")
    stack = np.stack([np.asarray(m) > 0 for m in final]) if final else np.zeros((0, H, W), bool)
    areas = stack.reshape(len(final), -1).sum(1) if final else np.zeros(0, np.int64)
    boxes = np.array([bbox_xyxy(m) for m in stack], np.int32).reshape(-1, 4)
    np.savez_compressed(fin / "npz" / f"{stem}.npz", masks=stack, area_px=areas, bbox_xyxy=boxes)
    make_overlay(image, final).save(out / "04_overlay" / f"{stem}.png")
    if SAVE_PER_CONFIG:
        for name, ms in per_cfg.items():
            for k, m in enumerate(ms):
                save_bin(m, out / "01_per_config" / stem / name / f"{k}.png")
    if SAVE_MERGED:
        for k, m in enumerate(merged):
            save_bin(m, out / "02_merged" / stem / f"{k}.png")
    meta = {"frame": stem, "image": str(img_path), "height": H, "width": W,
            "n_per_config": {n: len(v) for n, v in per_cfg.items()},
            "failed_configs": list(failed), "n_merged": len(merged), "n_final": len(final), "seconds": round(seconds, 2),
            "masks": [{"id": k, "file": f"03_final/masks/{stem}/{k}.png",
                       "area_px": int(areas[k]), "area_pct": round(100.0 * areas[k] / (H * W), 4),
                       "bbox_xyxy": boxes[k].tolist()} for k in range(len(final))]}
    (fin / "meta" / f"{stem}.json").write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")  # ghi cuối = dấu "đã xong"
    return meta

def process_scene(sc):
    """Chạy SAM (4 cấu hình) trên mọi ảnh của 1 scene. Trả về (số frame mới, số frame bỏ qua, giây)."""
    out, done, skipped, t_run = sc["out"], 0, 0, time.time()
    for img_path in tqdm(sc["images"], desc=sc["name"]):
        stem = img_path.stem
        if RESUME and (out / "03_final" / "meta" / f"{stem}.json").exists():
            skipped += 1
            continue
        t0 = time.time()
        image = np.array(Image.open(img_path).convert("RGB"))
        generated, per_cfg, failed = run_sam_configs(image)
        merged, final = combine_configs(generated)
        meta = write_frame(out, stem, img_path, image, per_cfg, merged, final, time.time() - t0, failed)
        done += 1
        tqdm.write(f"{sc['name']}/{stem}: {meta['n_final']} mask ({meta['seconds']}s) | per-config {meta['n_per_config']}"
                   + (f" | LỖI: {failed}" if failed else ""))
        del image, generated, per_cfg, merged, final
        torch.cuda.empty_cache()
    return done, skipped, time.time() - t_run

## 9. Hàm tổng hợp 1 scene: `manifest.json`, `stats.csv`, `instances.csv`, `run_info.json`, `palette.json`

In [ ]:
import importlib.metadata as im

def _ver(pkg):
    try: return im.version(pkg)
    except Exception: return None

def _sam2_commit():
    """Commit git của SAM 2 đã cài (uv ghi trong direct_url.json)."""
    for name in ("SAM-2", "sam-2", "sam2"):
        try:
            return json.loads(im.distribution(name).read_text("direct_url.json"))["vcs_info"]["commit_id"]
        except Exception:
            continue
    return None

def finalize_scene(sc, run_seconds, peak_gib):
    out = sc["out"]
    metas = [json.loads((out / "03_final" / "meta" / f"{s}.json").read_text(encoding="utf-8")) for s in sc["stems"]
             if (out / "03_final" / "meta" / f"{s}.json").exists()]
    assert len(metas) == len(sc["images"]), f"[{sc['name']}] mới có {len(metas)}/{len(sc['images'])} frame."
    sc["metas"] = metas

    manifest = {
        "scene": sc["name"], "image_dir": str(sc["image_dir"]), "num_frames": len(metas),
        "id_note": "ID mask chỉ có nghĩa trong 1 frame; id_map: 0 = nền, k+1 = mask k (mask nhỏ đè mask lớn).",
        "formats": {"masks": "03_final/masks/<frame>/<k>.png (0/255)", "id_map": "03_final/id_map/<frame>.png (uint16)",
                    "npz": "03_final/npz/<frame>.npz (masks[N,H,W] bool, area_px, bbox_xyxy)",
                    "overlay": "04_overlay/<frame>.png"},
        "frames": [{"frame": m["frame"], "image": m["image"], "height": m["height"], "width": m["width"],
                    "n_masks": m["n_final"], "mask_dir": f"03_final/masks/{m['frame']}",
                    "id_map": f"03_final/id_map/{m['frame']}.png", "npz": f"03_final/npz/{m['frame']}.npz",
                    "overlay": f"04_overlay/{m['frame']}.png"} for m in metas],
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    (out / "frames.txt").write_text("\n".join(m["frame"] for m in metas), encoding="utf-8")

    with open(out / "stats.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["frame", *[f"so_mask_{n}" for n in CFG_NAMES], "so_mask_sau_gop", "so_mask_cuoi",
                    "dien_tich_min_pct", "dien_tich_max_pct", "dien_tich_tb_pct", "giay", "cau_hinh_loi"])
        for m in metas:
            a = [x["area_pct"] for x in m["masks"]] or [0.0]
            w.writerow([m["frame"], *[m["n_per_config"].get(n, 0) for n in CFG_NAMES], m["n_merged"], m["n_final"],
                        f"{min(a):.3f}", f"{max(a):.3f}", f"{np.mean(a):.3f}", m["seconds"], ";".join(m.get("failed_configs", []))])

    with open(out / "instances.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["frame", "mask_id", "id_map_value", "area_px", "area_pct", "x0", "y0", "x1", "y1", "file"])
        for m in metas:
            for x in m["masks"]:
                w.writerow([m["frame"], x["id"], x["id"] + 1, x["area_px"], x["area_pct"], *x["bbox_xyxy"], x["file"]])

    max_n = max([m["n_final"] for m in metas] + [1])
    (out / "palette.json").write_text(json.dumps({str(i): id_color(i) for i in range(1, max_n + 1)}), encoding="utf-8")

    failed_frames = {n: sum(n in m.get("failed_configs", []) for m in metas) for n in CFG_NAMES}
    run_info = {
        "scene": sc["name"], "frames_with_failed_config": failed_frames, "image_dir": str(sc["image_dir"]), "frames": len(metas), "frame_stride": FRAME_STRIDE,
        "sam2_model": SAM2_MODEL, "sam2_checkpoint": ckpt_name, "sam2_config": MODEL_CFG, "autocast": str(amp_dtype),
        "gpu": torch.cuda.get_device_name(0), "sam_configs": SAM_CONFIGS,
        "merge_thresh": MERGE_THRESH, "dup_thresh": DUP_THRESH, "min_mask_ratio": MIN_MASK_RATIO,
        "seconds_this_run": round(run_seconds, 1), "vram_pytorch_peak_gib_this_run": round(peak_gib, 2),
        "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "torchvision": _ver("torchvision"),
                     "numpy": np.__version__, "pillow": _ver("pillow"), "hydra-core": _ver("hydra-core"),
                     "iopath": _ver("iopath"), "sam2": _ver("SAM-2") or _ver("sam2"), "sam2_commit": _sam2_commit(),
                     "pinned_author_env": CAN_PIN},
        "source": "Split_and_Splat/sam2/auto_seg.py (MaskMerging, useless_mask, 4 cấu hình SAM)",
    }
    (out / "run_info.json").write_text(json.dumps(run_info, indent=2, ensure_ascii=False), encoding="utf-8")
    return {"scene": sc["name"], "frames": len(metas), "masks": sum(m["n_final"] for m in metas),
            "masks_per_frame": round(float(np.mean([m["n_final"] for m in metas])), 1),
            "seconds_this_run": round(run_seconds, 1), "vram_peak_gib": round(peak_gib, 2),
            **{f"frames_loi_{n}": c for n, c in failed_frames.items()}}

## 10. Hàm nén / tải về máy

In [ ]:
def part_key(rel):
    """Tên frame mà file này thuộc về (None = file cấp cao nhất như manifest.json)."""
    p = rel.parts
    if p[0] in ("01_per_config", "02_merged"): return p[1]
    if p[0] == "03_final":                      return p[2] if p[1] == "masks" else Path(p[2]).stem
    if p[0] == "04_overlay":                    return Path(p[1]).stem
    return None

def zip_scene(sc, parts=1):
    """Nén 1 scene thành `parts` file zip (chia theo frame). Bên trong zip: <scene>/<cấu trúc thư mục kết quả>."""
    parts = max(1, int(parts))
    zip_dir = Path("/content") if Path("/content").is_dir() else Path(".")
    idx_of = {s: i * parts // len(sc["stems"]) for i, s in enumerate(sc["stems"])}
    zips = [zip_dir / (f"{sc['name']}_sam2_masks" + (f"_part{k + 1}of{parts}" if parts > 1 else "") + ".zip")
            for k in range(parts)]
    handles = [zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED, compresslevel=1) for z in zips]
    try:
        for f in sorted(sc["out"].rglob("*")):
            if f.is_file():
                rel = f.relative_to(sc["out"])
                handles[idx_of.get(part_key(rel), 0)].write(f, arcname=str(Path(sc["name"]) / rel))
    finally:
        for h in handles:
            h.close()
    for z in zips:
        mb = z.stat().st_size / 1e6
        print(f"  {z.name}: {mb:.0f} MB" + ("  ⚠️ lớn, nên tăng ZIP_PARTS" if mb > 1500 else ""))
    return zips

def download_scene(sc):
    zips = zip_scene(sc, ZIP_PARTS)
    from google.colab import files
    for z in zips:
        files.download(str(z))

## 11. Chạy toàn bộ: mọi scene × mọi ảnh × 4 cấu hình

Chạy lần lượt từng scene; xong scene nào thì tổng hợp file kết quả và (nếu `DOWNLOAD_WHEN = after_each_scene`) nén tải về máy ngay.
Trình duyệt có thể hỏi cho phép tải nhiều file — bấm Cho phép.

In [ ]:
SUMMARY = []
t_all = time.time()
for si, sc in enumerate(SCENES_LIST, 1):
    print(f"\n===== [{si}/{len(SCENES_LIST)}] {sc['name']}: {len(sc['images'])} ảnh × {len(SAM_CONFIGS)} cấu hình =====")
    torch.cuda.reset_peak_memory_stats()
    done, skipped, secs = process_scene(sc)
    peak = torch.cuda.max_memory_reserved() / 1024**3
    print(f"-> {sc['name']}: {done} frame mới, bỏ qua {skipped} | {secs:.0f}s | VRAM PyTorch đỉnh {peak:.2f} GiB")
    SUMMARY.append(finalize_scene(sc, secs, peak))
    if DOWNLOAD_WHEN == "after_each_scene":
        download_scene(sc)

summary_df = pd.DataFrame(SUMMARY)
summary_df.to_csv(Path(OUTPUT_ROOT) / "summary.csv", index=False, encoding="utf-8")
print(f"\nXong {len(SCENES_LIST)} scene trong {(time.time() - t_all) / 60:.1f} phút")
display(summary_df.set_index("scene"))

## 12. Xem kết quả: cây thư mục

In [ ]:
def tree(root, max_files=3):
    """In cây thư mục, mỗi thư mục chỉ liệt kê vài file đầu + tổng số."""
    root = Path(root)
    def walk(d, prefix=""):
        entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), natural_key(p)))
        dirs  = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.is_file()]
        shown_dirs = dirs if len(dirs) <= 4 else dirs[:2]
        for e in shown_dirs:
            n = sum(1 for _ in e.rglob("*") if _.is_file())
            print(f"{prefix}📁 {e.name}/  ({n} file)")
            walk(e, prefix + "    ")
        if len(dirs) > len(shown_dirs):
            print(f"{prefix}… và {len(dirs) - len(shown_dirs)} thư mục tương tự")
        for e in files[:max_files]:
            print(f"{prefix}📄 {e.name}  ({e.stat().st_size / 1024:.1f} KB)")
        if len(files) > max_files:
            print(f"{prefix}… và {len(files) - max_files} file nữa")
    print(f"📁 {root}")
    walk(root, "    ")

for sc in SCENES_LIST:
    tree(sc["out"])
    print()

## 13. Xem kết quả: ảnh, thống kê (mỗi scene vài frame)

In [ ]:
import matplotlib.pyplot as plt

for sc in SCENES_LIST:
    out, metas = sc["out"], sc["metas"]
    pick = [metas[i] for i in np.linspace(0, len(metas) - 1, min(3, len(metas))).astype(int)]
    fig, axes = plt.subplots(len(pick), 3, figsize=(15, 4.2 * len(pick)), squeeze=False)
    for r, m in enumerate(pick):
        img = Image.open(m["image"]).convert("RGB")
        ov = Image.open(out / "04_overlay" / f"{m['frame']}.png")
        idm = np.array(Image.open(out / "03_final" / "id_map" / f"{m['frame']}.png"))
        col = np.zeros((*idm.shape, 3), np.uint8)
        for v in np.unique(idm):
            if v: col[idm == v] = id_color(int(v))
        for c, (im_, t) in enumerate([(img, "ảnh gốc"), (ov, f"overlay ({m['n_final']} mask)"), (col, "id_map")]):
            axes[r, c].imshow(im_); axes[r, c].set_title(f"{sc['name']}/{m['frame']} — {t}"); axes[r, c].axis("off")
    plt.tight_layout(); plt.show()

    stats = pd.read_csv(out / "stats.csv")
    inst = pd.read_csv(out / "instances.csv")
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    stats.plot(x="frame", y=[f"so_mask_{n}" for n in CFG_NAMES] + ["so_mask_cuoi"], ax=ax[0],
               title=f"{sc['name']}: số mask / frame (từng cấu hình → cuối)")
    ax[0].set_xticks([]); ax[0].set_xlabel("frame")
    if len(inst):
        ax[1].hist(inst["area_pct"], bins=40); ax[1].set_xscale("log")
    ax[1].set_title("Phân bố diện tích mask (% ảnh)")
    plt.tight_layout(); plt.show()

## 14. Kiểm tra tính nhất quán của output

In [ ]:
all_problems = []
for sc in SCENES_LIST:
    out, problems = sc["out"], []
    for m in sc["metas"]:
        s, H, W = m["frame"], m["height"], m["width"]
        files = sorted((out / "03_final" / "masks" / s).glob("*.png"))
        idm = np.array(Image.open(out / "03_final" / "id_map" / f"{s}.png"))
        with np.load(out / "03_final" / "npz" / f"{s}.npz") as z:
            npz_shape = z["masks"].shape
        if len(files) != m["n_final"]:                problems.append((s, "số file mask ≠ meta"))
        if idm.shape != (H, W):                       problems.append((s, "id_map sai kích thước"))
        if npz_shape != (m["n_final"], H, W):         problems.append((s, "npz sai shape"))
        if idm.max(initial=0) > m["n_final"]:         problems.append((s, "id_map có ID vượt số mask"))
        if files and np.array(Image.open(files[0])).shape != (H, W): problems.append((s, "mask sai kích thước"))
    all_problems += [(sc["name"], *p) for p in problems]
    print(f"{sc['name']}: " + ("OK — tất cả khớp" if not problems else f"{len(problems)} vấn đề") +
          f" | {len(sc['metas'])} frame, {sum(m['n_final'] for m in sc['metas'])} mask")
for p in all_problems[:20]:
    print(" ", p)

## 15. Cách các model khác đọc mask chung

Mọi model chỉ cần `manifest.json` + 3 hàm dưới đây (copy sang script train của model đó, không cần SAM, không cần GPU).

In [ ]:
def load_manifest(root):
    return json.loads((Path(root) / "manifest.json").read_text(encoding="utf-8"))

def load_masks(root, frame):
    """(N,H,W) bool — mỗi lát là 1 mask, N có thể = 0."""
    with np.load(Path(root) / "03_final" / "npz" / f"{frame}.npz") as z:
        return z["masks"]

def load_id_map(root, frame):
    """(H,W) uint16 — 0 = nền, k+1 = mask k. Tiện làm nhãn cho mask loss khi train 3DGS."""
    return np.array(Image.open(Path(root) / "03_final" / "id_map" / f"{frame}.png"))

# Ví dụ: root = thư mục của 1 scene
root = SCENES_LIST[0]["out"]
mf = load_manifest(root)
fr = mf["frames"][0]["frame"]
print(mf["scene"], fr, "→ masks", load_masks(root, fr).shape, "| id_map", load_id_map(root, fr).shape, load_id_map(root, fr).dtype)

## 16. Tải về máy (nếu chọn `at_end`)

In [ ]:
if DOWNLOAD_WHEN == "at_end":
    for sc in SCENES_LIST:
        print(sc["name"])
        download_scene(sc)
else:
    print(f"DOWNLOAD_WHEN = {DOWNLOAD_WHEN!r}: không tải ở bước này.")

## Bước tiếp theo: ID nhất quán giữa các view (Stage 2 của Split&Splat)

Notebook này cho **mask 2D từng frame độc lập**. Để dựng 3DGS cho **từng object** thì cùng một vật phải mang cùng ID ở mọi view. Tác giả làm việc này trong `sam2/mask_propagation_scanet.py` bằng depth + pose + point cloud 3D. Code đó đọc cố định:

```
data/<scene>/images/<số>.JPEG              # tên file phải là SỐ (code gọi int(tên)); ảnh LERF tên frame_00001.jpg cần bỏ tiền tố
data/<scene>/depth/<số>.png                # uint16, đơn vị mm
data/<scene>/pose/<số>.txt                 # camera→world 4x4
data/<scene>/intrinsic/intrinsic_depth.txt # 4x4
data/<scene>/<scene>_vh_clean_2.ply        # point cloud 3D cùng hệ toạ độ với pose
output/<scene>_autoseg_mask/<frame>/<k>.png   ← chính là thư mục 03_final/masks của notebook này
```

Vì vậy chỉ cần: đặt tên ảnh đầu vào là số (`00001.jpg`…), rồi trỏ/copy `03_final/masks` vào `output/<scene>_autoseg_mask`.
Stage 2 cần thêm depth, pose và point cloud nên chưa nằm trong notebook này; nó cũng phải build extension CUDA `point_projection`.